In [10]:
import pandas as pd

df_ip = pd.read_csv('../data/raw/IpAddress_to_Country.csv')
print("IP database columns:", df_ip.columns.tolist())
print("\nFirst 5 rows:")
print(df_ip.head())
print("\nSample lower_bound_ip_address:", df_ip['lower_bound_ip_address'].iloc[0])
print("Type:", type(df_ip['lower_bound_ip_address'].iloc[0]))

IP database columns: ['lower_bound_ip_address', 'upper_bound_ip_address', 'country']

First 5 rows:
   lower_bound_ip_address  upper_bound_ip_address    country
0              16777216.0                16777471  Australia
1              16777472.0                16777727      China
2              16777728.0                16778239      China
3              16778240.0                16779263  Australia
4              16779264.0                16781311      China

Sample lower_bound_ip_address: 16777216.0
Type: <class 'numpy.float64'>


In [11]:
import pandas as pd
import numpy as np

# Load data
df_fraud = pd.read_csv('../data/raw/Fraud_Data.csv')
df_ip = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print(f"Fraud data: {df_fraud.shape}")
print(f"IP data: {df_ip.shape}")

# Convert to integer (remove .0)
df_fraud['ip_int'] = df_fraud['ip_address'].astype('int64')
df_ip['lower_int'] = df_ip['lower_bound_ip_address'].astype('int64')
df_ip['upper_int'] = df_ip['upper_bound_ip_address'].astype('int64')

print(f"Fraud valid IPs: {len(df_fraud)}")
print(f"IP range records: {len(df_ip)}")
print(f"Fraud IP range: {df_fraud['ip_int'].min()} to {df_fraud['ip_int'].max()}")
print(f"IP DB range: {df_ip['lower_int'].min()} to {df_ip['upper_int'].max()}")

# Sort for merge_asof
df_ip_sorted = df_ip.sort_values('lower_int')
df_fraud_sorted = df_fraud.sort_values('ip_int')

# Merge
df_merged = pd.merge_asof(df_fraud_sorted, df_ip_sorted, left_on='ip_int', right_on='lower_int', direction='backward')

# Filter where ip_int <= upper_int
df_merged = df_merged[df_merged['ip_int'] <= df_merged['upper_int']]

print(f"Final merged shape: {len(df_merged)}")

if len(df_merged) > 0:
    # Keep needed columns
    keep_cols = ['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id',
                 'source', 'browser', 'sex', 'age', 'class', 'country']
    df_merged = df_merged[keep_cols]
    df_merged.to_csv('../data/processed/fraud_data_with_country.csv', index=False)
    print(f"Saved {len(df_merged)} rows")
    print("\nSample:")
    print(df_merged.head())
else:
    print("No matches found. Check if IP ranges cover the fraud IPs.")

Fraud data: (151112, 11)
IP data: (138846, 3)
Fraud valid IPs: 151112
IP range records: 138846
Fraud IP range: 52093 to 4294850499
IP DB range: 16777216 to 3758096383
Final merged shape: 129146
Saved 129146 rows

Sample:
     user_id          signup_time        purchase_time  purchase_value  \
634   247547  2015-06-28 03:00:34  2015-08-09 03:57:29              47   
635   220737  2015-01-28 14:21:11  2015-02-11 20:28:28              15   
636   390400  2015-03-19 20:49:09  2015-04-11 23:41:23              44   
637    69592  2015-02-24 06:11:57  2015-05-23 16:40:14              55   
638   174987  2015-07-07 12:58:11  2015-11-03 04:04:30              51   

         device_id  source browser sex  age  class    country  
634  KIXYSVCHIPQBR     SEO  Safari   F   30      0  Australia  
635  PKYOWQKWGJNJI     SEO  Chrome   F   34      0   Thailand  
636  LVCSXLISZHVUO     Ads      IE   M   29      0      China  
637  UHAUHNXXUADJE  Direct  Chrome   F   30      0      China  
638  XPGPMOHID

In [12]:
import pandas as pd
import numpy as np

# Load the enriched data
df = pd.read_csv('../data/processed/fraud_data_with_country.csv')

# Convert time columns
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

# Time since signup (in hours)
df['time_since_signup'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 3600

# Hour of day
df['purchase_hour'] = df['purchase_time'].dt.hour

# Day of week
df['purchase_dayofweek'] = df['purchase_time'].dt.dayofweek

# Transaction velocity (count per user)
df['transaction_count'] = df.groupby('user_id').cumcount() + 1

print(f"Shape after feature engineering: {df.shape}")
print("\nNew features added:")
print(f"- time_since_signup: {df['time_since_signup'].min():.0f} to {df['time_since_signup'].max():.0f} hours")
print(f"- purchase_hour: {df['purchase_hour'].min()} to {df['purchase_hour'].max()}")
print(f"- purchase_dayofweek: {df['purchase_dayofweek'].min()} to {df['purchase_dayofweek'].max()}")

Shape after feature engineering: (129146, 15)

New features added:
- time_since_signup: 0 to 2880 hours
- purchase_hour: 0 to 23
- purchase_dayofweek: 0 to 6


In [13]:
# One-hot encode categorical columns
categorical_cols = ['source', 'browser', 'sex', 'country']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {df_encoded.shape}")
print(f"Columns: {df_encoded.columns.tolist()[:15]}...")

Shape after encoding: (129146, 198)
Columns: ['user_id', 'signup_time', 'purchase_time', 'purchase_value', 'device_id', 'age', 'class', 'time_since_signup', 'purchase_hour', 'purchase_dayofweek', 'transaction_count', 'source_Direct', 'source_SEO', 'browser_FireFox', 'browser_IE']...


In [14]:
# Drop columns not needed for modeling
drop_cols = ['user_id', 'signup_time', 'purchase_time', 'device_id']
df_final = df_encoded.drop(columns=drop_cols)

print(f"Final feature shape: {df_final.shape}")
print(f"Target column: class")
print(f"Class distribution:\n{df_final['class'].value_counts()}")

Final feature shape: (129146, 194)
Target column: class
Class distribution:
class
0    116878
1     12268
Name: count, dtype: int64


In [15]:
df_final.to_csv('../data/processed/fraud_data_processed.csv', index=False)
print("Saved to ../data/processed/fraud_data_processed.csv")

Saved to ../data/processed/fraud_data_processed.csv


In [16]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Split data
X = df_final.drop('class', axis=1)
y = df_final['class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE:")
print(f"  Train shape: {X_train.shape}")
print(f"  Train fraud: {y_train.sum()} ({y_train.mean()*100:.4f}%)")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Train shape: {X_train_resampled.shape}")
print(f"  Train fraud: {y_train_resampled.sum()} ({y_train_resampled.mean()*100:.4f}%)")

Before SMOTE:
  Train shape: (103316, 193)
  Train fraud: 9814 (9.4990%)

After SMOTE:
  Train shape: (187004, 193)
  Train fraud: 93502 (50.0000%)


In [17]:
import pandas as pd

# Load the processed data
df = pd.read_csv('../data/processed/fraud_data_processed.csv')
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['class'].value_counts())
print(f"Fraud percentage: {df['class'].mean()*100:.4f}%")

Final dataset shape: (129146, 194)
Columns: ['purchase_value', 'age', 'class', 'time_since_signup', 'purchase_hour', 'purchase_dayofweek', 'transaction_count', 'source_Direct', 'source_SEO', 'browser_FireFox', 'browser_IE', 'browser_Opera', 'browser_Safari', 'sex_M', 'country_Albania', 'country_Algeria', 'country_Angola', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Bangladesh', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Benin', 'country_Bermuda', 'country_Bhutan', 'country_Bolivia', 'country_Bonaire; Sint Eustatius; Saba', 'country_Bosnia and Herzegowina', 'country_Botswana', 'country_Brazil', 'country_British Indian Ocean Territory', 'country_Brunei Darussalam', 'country_Bulgaria', 'country_Burkina Faso', 'country_Burundi', 'country_Cambodia', 'country_Cameroon', 'country_Canada', 'country_Cape Verde', 'countr